# RP-MPQ v2: Rate-adaptive Precision Mixed-Precision Quantization

This notebook runs the `rpmpq_v2.py` pipeline step by step on Google Colab.

## Pipeline Overview

| Step | Command | GPU? | Approx. Time | Description |
|------|---------|------|-------------|-------------|
| **1. Collect** | `--step collect` | **Yes (GPU)** | ~30-60 min | Runs inference for every policy in the LUT on all test samples. Computes per-sample NMSE, rates, and zeta-proxy values. Saves `rpmpq_v2_collected.csv`, `rpmpq_v2_zeta.csv`, `rpmpq_v2_fp32_ref.csv`, `rpmpq_v2_perfect_rates.csv`. |
| **2. Build** | `--step build` | No (CPU) | ~1 min | Constructs 2D importance surface Omega(zeta, SNR), optimises per-state policy assignments. Saves `rpmpq_v2_omega.npz`, `rpmpq_v2_kappa.csv`, `rpmpq_v2_policy_map.csv`. |
| **3. Eval** | `--step eval` | **Yes (GPU)** | ~10-20 min | Evaluates all schemes: uniform INT16/8/4, static MP, online Hoyer, online zeta, FP32 reference. Saves `rpmpq_v2_eval_summary.csv` and generates comparison plots. |
| **4. Plot** | `--step plot` | No (CPU) | ~10 sec | Re-generates plots from cached results (importance surface, summary table). Useful for tweaking plot aesthetics without re-running eval. |

## Output Files

All CSV/NPZ files go to `results/csv/`, all plots go to `results/plots/`.

**Key plots:**
- `rpmpq_v2_nmse_comparison.png` — NMSE (dB) comparison across all schemes
- `rpmpq_v2_outage.png` — Outage probability curves per SNR
- `rpmpq_v2_conditional_nmse.png` — NMSE conditioned on zeta bins
- `rpmpq_v2_importance_surface.png` — 2D importance heatmap Omega(zeta, SNR)

## Prerequisites
- Model checkpoint at `saved_models/mamba_transnet_L2_dim512_baseline/best.pth`
- COST2100 data at `data/DATA_Htrainout.mat` and `data/DATA_Htestout.mat`
- Policy LUT at `results/csv/mp_policy_lut_mamba_pruned.csv` (or `mp_policy_lut_mamba_wide_pruned.csv`)

## Cell 1: Environment Setup

In [ ]:
# Mount Google Drive + install dependencies
import os, sys

PROJECT_ROOT = "/content/drive/MyDrive/MambaCompression"
MAMBAIC_ROOT = os.path.join(PROJECT_ROOT, "MambaIC")

if not os.path.isdir(PROJECT_ROOT):
    from google.colab import drive
    drive.mount('/content/drive')

assert os.path.isdir(MAMBAIC_ROOT), f"MambaIC not found: {MAMBAIC_ROOT}"
os.chdir(MAMBAIC_ROOT)
print(f"Working directory: {os.getcwd()}")

# Run the cached kernel setup (VMamba CUDA kernel + core deps)
setup_path = os.path.join(PROJECT_ROOT, "setup_colab.py")
if os.path.isfile(setup_path):
    print("Running setup_colab.py ...")
    exec(open(setup_path).read())
else:
    print("setup_colab.py not found, installing deps manually ...")
    !pip install -q einops scipy tqdm thop fvcore pybind11

# Additional deps for rpmpq_v2.py
!pip install -q pulp hilbertcurve seaborn compressai timm 2>/dev/null | tail -1

print("\n=== Environment Ready ===")

## Cell 2: Quick Sanity Check

In [ ]:
# Verify imports and basic functions work
import sys, os, numpy as np
sys.path.insert(0, '.')

from rpmpq_v2 import build_kernel, compute_zeta_proxy
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Steps 1 and 3 will be very slow.")

# Test kernel construction
K = build_kernel(32, 1.0)
print(f"\nKernel OK: shape={K.shape}, row_sum={K.sum(1)[0]:.4f}")

# Test zeta computation on random input
x_dummy = np.random.rand(2, 32, 32).astype(np.float32)
zeta = compute_zeta_proxy(x_dummy, K, K)
print(f"Zeta proxy (random input): {zeta:.4f}")

# Check required files
required = {
    "Checkpoint": "saved_models/mamba_transnet_L2_dim512_baseline/best.pth",
    "Train data": "data/DATA_Htrainout.mat",
    "Test data":  "data/DATA_Htestout.mat",
}
for name, path in required.items():
    exists = os.path.isfile(path)
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {name}: {path}")

# Check policy LUT (auto-detected)
lut_candidates = [
    "results/csv/mp_policy_lut_mamba_pruned.csv",
    "results/csv/mp_policy_lut_mamba_wide_pruned.csv",
]
lut_found = False
for lut in lut_candidates:
    if os.path.isfile(lut):
        print(f"  [OK] Policy LUT: {lut}")
        lut_found = True
        break
if not lut_found:
    print("  [MISSING] Policy LUT: run train_ae.py --analyze_all first")

print("\nSanity check complete.")

## Cell 3: Step 1 — Collect Per-Sample Data (GPU, ~30-60 min)

This is the **most expensive step**. For every policy in the pruned LUT and every test sample, it:
1. Applies the mixed-precision weight quantization policy
2. Runs encoder + feedback quantization + decoder
3. Records per-sample NMSE (linear) and sum-rate at each SNR
4. Computes zeta-proxy (energy dispersion descriptor) for each sample

**Outputs:**
- `results/csv/rpmpq_v2_collected.csv` — per-sample per-policy performance
- `results/csv/rpmpq_v2_zeta.csv` — zeta proxy values per sample
- `results/csv/rpmpq_v2_fp32_ref.csv` — FP32 reference performance
- `results/csv/rpmpq_v2_perfect_rates.csv` — perfect CSI rates (upper bound)

In [ ]:
!python rpmpq_v2.py --step collect

## Cell 4: Step 2 — Build Importance Surface (CPU, ~1 min)

Constructs the 2D importance surface Omega(zeta, SNR) and optimises per-state policy assignments via the shrinkage estimator.

**Outputs:**
- `results/csv/rpmpq_v2_omega.npz` — importance surface + bin edges
- `results/csv/rpmpq_v2_kappa.csv` — per-state kappa values
- `results/csv/rpmpq_v2_policy_map.csv` — optimal policy per (zeta_bin, snr_bin)
- `results/plots/rpmpq_v2_importance_surface.png` — heatmap visualisation

In [ ]:
!python rpmpq_v2.py --step build

## Cell 5: Step 3 — Evaluate All Schemes (GPU, ~10-20 min)

Evaluates and compares the following quantization schemes:
- **Uniform:** INT16, INT8, INT4 (all layers same bit-width)
- **Static MP:** Single best mixed-precision policy (offline)
- **Online Hoyer:** RP-MPQ v1 — selects policy based on Hoyer sparsity
- **Online Zeta:** RP-MPQ v2 — selects policy based on zeta-proxy + SNR state
- **FP32:** Full-precision reference (upper bound)

**Outputs:**
- `results/csv/rpmpq_v2_eval_summary.csv` — aggregated metrics table
- `results/csv/rpmpq_v2_budget_consistency.csv` — saving vs NMSE per scheme
- `results/plots/rpmpq_v2_nmse_comparison.png`
- `results/plots/rpmpq_v2_outage.png`
- `results/plots/rpmpq_v2_conditional_nmse.png`

In [ ]:
!python rpmpq_v2.py --step eval

## Cell 6: Step 4 — Generate Plots (CPU)

Re-generates plots from cached results without re-running inference. Useful for adjusting aesthetics.

In [ ]:
!python rpmpq_v2.py --step plot

## Cell 7: View Results

In [ ]:
import os, glob
import pandas as pd
from IPython.display import Image, display

# ---- Print evaluation summary table ----
summary_csv = 'results/csv/rpmpq_v2_eval_summary.csv'
if os.path.isfile(summary_csv):
    print("=" * 60)
    print("  Evaluation Summary")
    print("=" * 60)
    df = pd.read_csv(summary_csv)
    display(df)
else:
    print("No eval summary found. Run steps 1-3 first.")

# ---- Print budget consistency table ----
budget_csv = 'results/csv/rpmpq_v2_budget_consistency.csv'
if os.path.isfile(budget_csv):
    print("\n" + "=" * 60)
    print("  Budget Consistency")
    print("=" * 60)
    display(pd.read_csv(budget_csv))

# ---- Display all plots ----
plot_dir = 'results/plots'
plot_files = sorted(glob.glob(f'{plot_dir}/rpmpq_v2_*.png'))
if plot_files:
    print(f"\nFound {len(plot_files)} plots:")
    for f in plot_files:
        print(f'\n--- {os.path.basename(f)} ---')
        display(Image(f))
else:
    print("\nNo plots found yet. Run steps 1-3 first.")

## Cell 8: (Optional) Run Everything at Once

Runs all four steps sequentially. Total time: ~40-80 min depending on GPU.

In [ ]:
# Alternative: run the full pipeline in one shot
!python rpmpq_v2.py --step all

## Cell 9: (Optional) Indoor Scenario

By default, the script uses outdoor data (`DATA_Htrainout.mat` / `DATA_Htestout.mat`). To run on indoor data, override the paths explicitly:

In [ ]:
# Indoor scenario — override train/test paths
# !python rpmpq_v2.py --step all \
#     --train_path data/DATA_Htrainin.mat \
#     --test_path data/DATA_Htestin.mat

## Cell 10: (Optional) Custom Parameters

The script exposes many tunable parameters. Here are the key ones:

```
# Model / data
--encoder mamba          # encoder architecture
--decoder transnet       # decoder architecture
--encoded_dim 512        # bottleneck dimension
--checkpoint PATH        # override model checkpoint path
--policy_lut PATH        # override policy LUT CSV path

# Zeta parameters
--alpha_d 1.0            # kernel decay (delay domain)
--alpha_a 1.0            # kernel decay (angular domain)
--lambda_d 0.5           # proxy weight (delay coherence)
--lambda_a 0.5           # proxy weight (angular coherence)

# Importance surface
--J_bins 3               # number of SNR bins
--K_bins 10              # number of zeta bins
--tau_shr 10.0           # shrinkage parameter
--lambda_mult 1.0        # Lagrange multiplier for policy optimisation

# Evaluation
--snr_list 10 20 30      # SNR values to evaluate
--gamma_list 0.99 0.98 0.95  # outage thresholds
--aq 8                   # feedback (latent z) quantisation bits
--act_quant 16           # activation quantisation bits (fixed)
--batch_size 256         # inference batch size
```